# Carpet diagram — lgm temperature benchmarking

Step toward an update of IPCC AR5 WG1 Fig. 9.12 (a *portrait plot* / *carpet
diagram*). For each PMIP lgm (Last Glacial Maximum (21 ka)) simulation we:

1. Load the annual-mean near-surface temperature field `tas_spatialmean_ann` from the
   CVDP output, and the matching `piControl` run.
2. Compute the **lgm − piControl anomaly** on each model's native grid.
3. **Sample** that anomaly field at the location of every proxy reconstruction.
4. Summarise the model–data mismatch as the **root-mean-squared error (RMSE)** against
   each reconstruction compilation (Bartlein 21 ka (Bartlein et al. 2011), Cleator (Cleator et al. 2020),
                      Osman (LGMR; Osman et al. 2021) and Annan (Annan et al. 2022)).

Intermediate tables are written to `output/`. Run with the `my-cli-py` conda env.

In [1]:
import os, glob, re
import numpy as np
import pandas as pd
import xarray as xr

ROOT      = os.getcwd()  # run the notebook from the carpet_diagram/ directory
CVDP_DIR  = os.path.join(ROOT, 'cvdp_output_by_experiment')
EXPERIMENT = 'lgm'
RECON_DIR = os.path.join(ROOT, 'recons', EXPERIMENT)
OUT_DIR   = os.path.join(ROOT, 'output')
os.makedirs(OUT_DIR, exist_ok=True)

VAR        = 'tas_spatialmean_ann'
print('experiment:', EXPERIMENT)
print('CVDP   :', CVDP_DIR)
print('recons :', RECON_DIR)
print('output :', OUT_DIR)

experiment: lgm
CVDP   : /home/ucfaccb@ad.ucl.ac.uk/Documents/local_repos/PMIP7-vision/carpet_diagram/cvdp_output_by_experiment
recons : /home/ucfaccb@ad.ucl.ac.uk/Documents/local_repos/PMIP7-vision/carpet_diagram/recons/lgm
output : /home/ucfaccb@ad.ucl.ac.uk/Documents/local_repos/PMIP7-vision/carpet_diagram/output


## Step 1 — Reconstruction compilations

Five LGM (21 ka) compilations. Two are proxy compilations, with unit weights:

- **Bartlein** — the Bartlein et al. (2011) pollen-based mean-annual-temperature synthesis
  at 21 ka (`Bartlein_mat_21ka.csv`, the NOAA `mat_delta_21ka_ALL_grid_2x2.csv`), the 21 ka
  counterpart of the file used for the midHolocene row. 98 cells with an anomaly, in the
  `mat_anm_mean` column. Note this is the proxy compilation that Cleator assimilated, so
  the Cleator row below is not independent of it.
- **P2F** — the recommended annual sea-surface-temperature records assembled for the
  Past2Future synthesis (`P2F_recommended_sst_lgm.csv`, a verbatim CSV dump of the
  supplied workbook). The `SST_Anomaly` column is each core's annual SST minus the
  ERSSTv6 1870–1899 mean at its grid cell, so it is already an anomaly. These are SSTs
  compared against model `tas`, on the same footing as the lig127k and midPliocene
  marine rows.

The other three are near-global gridded data assimilation products, so each point gets a
`cos(latitude)` weight for an area-fair RMSE:

- **Cleator** — the Cleator et al. (2020) vegetation-model-inversion **data assimilation**
  product (its own metadata: "made by combining pollen based reconstructions (from Bartlein
  et al. 2011) and averaged outputs of LGM simulations from … PMIP … under a variational
  data assimilation technique"). A long metadata preamble precedes a `#`-prefixed header row
  (line 77); the only relevant field is `MAT` (mean-annual-temperature anomaly, °C).
- **Osman** — the Last Glacial Maximum Reanalysis (Osman et al. 2021), the ensemble-mean
  `sat` field from `Osman_LGMR_21ka_SAT_anom_climo.nc`, already differenced to a 21 ka −
  (0–1 ka) anomaly. The 96×144 grid is flattened to points, dropping fill values.
- **Annan** — Annan et al. (2022), an ensemble Kalman filter blending proxies with a
  19-member PMIP prior. The `SAT.mean` field of `Annan_etal22.SAT.mean.lgm.nc` is already
  an LGM anomaly: its area-weighted global mean is −4.46 °C, matching the −4.5 ± 0.9 °C
  the paper reports. Two quirks — the coordinates are named `latitude`/`longitude` rather
  than `lat`/`lon`, and the longitude axis is rolled (it runs 181…359 then 1…179 rather
  than ascending). The roll is harmless here because Step 1 flattens the grid to points,
  each carrying its own longitude, rather than interpolating the field.

In [2]:
# --- Bartlein: the 21 ka pollen MAT synthesis (scattered 2x2 cells) ---
bar = pd.read_csv(os.path.join(RECON_DIR, 'Bartlein_mat_21ka.csv'))
bar.columns = [c.strip() for c in bar.columns]   # headers carry leading spaces
bartlein = pd.DataFrame({
    'compilation': 'Bartlein', 'reference': 'Bartlein et al. 2011', 'site': np.nan,
    'Proxy': 'pollen MAT',
    'Latitude': bar['lat'].astype(float), 'Longitude': bar['lon'].astype(float),
    'Anom': bar['mat_anm_mean'].astype(float),
})
bartlein['source_table'] = 'Bartlein_mat_21ka.csv'
bartlein['weight'] = 1.0                  # scattered sites: equal weight

# --- Cleator: gridded land MAT anomalies from a CSV with a preamble ---
CLE_COLS = ['lat', 'lon', 'MI', 'MAP', 'MAT', 'MTCO', 'MTWA', 'GDD5',
            'MI_SD', 'MAP_SD', 'MAT_SD', 'MTCO_SD', 'MTWA_SD', 'GDD5_SD']
cle = pd.read_csv(os.path.join(RECON_DIR, 'cleator2020_recon.csv'),
                  skiprows=77, header=None, names=CLE_COLS)
cle = cle.apply(pd.to_numeric, errors='coerce').dropna(subset=['lat', 'lon', 'MAT'])
cleator = pd.DataFrame({
    'compilation': 'Cleator', 'reference': 'Cleator et al. 2020', 'site': np.nan,
    'Proxy': 'pollen MAT (assim.)',
    'Latitude': cle['lat'].astype(float), 'Longitude': cle['lon'].astype(float),
    'Anom': cle['MAT'].astype(float),
})
cleator['source_table'] = 'cleator2020_recon.csv'

# --- Osman: flatten the gridded LGMR SAT anomaly field to points ---
ds = xr.open_dataset(os.path.join(RECON_DIR, 'Osman_LGMR_21ka_SAT_anom_climo.nc'),
                     decode_times=False)
sat = ds['sat'].where(np.abs(ds['sat']) < 1e30)
lon2d, lat2d = np.meshgrid(sat['lon'].values, sat['lat'].values)
osman = pd.DataFrame({
    'compilation': 'Osman', 'reference': 'Osman et al. 2021', 'site': np.nan,
    'Proxy': 'LGMR SAT (assim.)',
    'Latitude': lat2d.ravel().astype(float), 'Longitude': lon2d.ravel().astype(float),
    'Anom': sat.values.ravel().astype(float),
})
osman['source_table'] = 'Osman_LGMR_21ka_SAT_anom_climo.nc'

# --- Annan: flatten the gridded EnKF SAT anomaly field to points ---
# Coordinates here are 'latitude'/'longitude', and the longitude axis is rolled
# (181..359 then 1..179). That does not matter: we flatten to points, so every
# value keeps its own longitude and no interpolation of this field takes place.
ANNAN_FILE = 'Annan_etal22.SAT.mean.lgm.nc'
ads = xr.open_dataset(os.path.join(RECON_DIR, ANNAN_FILE), decode_times=False)
asat = ads['SAT.mean'].where(np.abs(ads['SAT.mean']) < 1e30)
alon2d, alat2d = np.meshgrid(asat['longitude'].values, asat['latitude'].values)
annan = pd.DataFrame({
    'compilation': 'Annan', 'reference': 'Annan et al. 2022', 'site': np.nan,
    'Proxy': 'EnKF SAT (assim.)',
    'Latitude': alat2d.ravel().astype(float), 'Longitude': alon2d.ravel().astype(float),
    'Anom': asat.values.ravel().astype(float),
})
annan['source_table'] = ANNAN_FILE

for gridded in (cleator, osman, annan):
    # Regular lat/lon grids: cos-latitude weight makes the RMSE area-fair.
    gridded['weight'] = np.cos(np.deg2rad(gridded['Latitude']))


# --- P2F: the recommended annual SST records (scattered marine cores) ---
# A verbatim CSV dump of the P2F workbook (see recons/README for the command).
# `SST_Anomaly` is the core's annual SST minus the ERSSTv6 1870-1899 mean at its
# grid cell, so it is already the anomaly we want and needs no re-referencing.
# These are SSTs benchmarked against model `tas`, exactly as the lig127k Hoffman
# and Capron and the midPliocene Foley-Dowsett rows already are.
P2F_FILE = 'P2F_recommended_sst_lgm.csv'
p2f = pd.read_csv(os.path.join(RECON_DIR, P2F_FILE))
p2f.columns = [c.strip() for c in p2f.columns]

# Coordinate check: the site `Latitude` should agree with the spreadsheet's own
# ERSST grid latitude. Where the two disagree in sign the site column has a sign
# typo (e.g. GeoB7112-5, a Chile-margin core listed at +24 deg N), so take the
# hemisphere from the ERSST cell. Rows within 2 deg of the equator are skipped:
# there a sign difference is just rounding across it.
flip = (np.sign(p2f['Latitude']) != np.sign(p2f['ERSST Lat'])) & (p2f['ERSST Lat'].abs() > 2)
if flip.any():
    print('P2F: latitude sign taken from the ERSST cell for',
          ', '.join(p2f.loc[flip, 'Core'].astype(str)))
    p2f.loc[flip, 'Latitude'] = -p2f.loc[flip, 'Latitude']

p2f_pts = pd.DataFrame({
    'compilation': 'P2F', 'reference': 'P2F recommended SST records',
    'site': p2f['Core'].astype(str),
    'Proxy': p2f['Proxy/Proxies'].astype(str),
    'Latitude': p2f['Latitude'].astype(float),
    'Longitude': p2f['Longitude'].astype(float),
    'Anom': p2f['SST_Anomaly'].astype(float),
})
p2f_pts['source_table'] = P2F_FILE
p2f_pts['weight'] = 1.0                   # scattered sites: equal weight

recon = pd.concat([bartlein, cleator, osman, annan, p2f_pts], ignore_index=True)
recon = recon.dropna(subset=['Latitude', 'Longitude', 'Anom'])

print(recon.groupby('compilation').size())
recon_out = os.path.join(OUT_DIR, f'recon_points_{EXPERIMENT}.csv')
recon.to_csv(recon_out, index=False)
print('wrote', recon_out)
recon.head()

compilation
Annan       16200
Bartlein       98
Cleator      2214
Osman       13824
P2F           116
dtype: int64


wrote /home/ucfaccb@ad.ucl.ac.uk/Documents/local_repos/PMIP7-vision/carpet_diagram/output/recon_points_lgm.csv


,compilation,reference,site,Proxy,Latitude,Longitude,Anom,source_table,weight
0,Bartlein,Bartlein et al. 2011,NaN,pollen MAT,69.0,-161.0,2.3591,Bartlein_mat_21ka.csv,1.0
1,Bartlein,Bartlein et al. 2011,NaN,pollen MAT,67.0,-157.0,-3.0103,Bartlein_mat_21ka.csv,1.0
2,Bartlein,Bartlein et al. 2011,NaN,pollen MAT,67.0,-153.0,-4.4026,Bartlein_mat_21ka.csv,1.0
3,Bartlein,Bartlein et al. 2011,NaN,pollen MAT,67.0,-149.0,13.3879,Bartlein_mat_21ka.csv,1.0
4,Bartlein,Bartlein et al. 2011,NaN,pollen MAT,65.0,-147.0,-3.3921,Bartlein_mat_21ka.csv,1.0


## Step 2 — Model lgm − piControl anomalies

Each model writes its CVDP field on its own native grid, so anomalies are computed
per model. We pair every `lgm` file with the same model's `piControl` file
(the main file is the one named `<model>_<experiment>.cvdp_data.<years>.nc`; auxiliary
per-variable files such as `.siconc.` / `.zos.` / `.monsoon.` / `.tas.indices.` carry a
extra token before the years and are skipped). If the two grids ever differ, the control
is bilinearly regridded onto the lgm grid before differencing.

In [3]:
def model_files(experiment):
    """Map model name -> path for the main CVDP file of an experiment.

    Only `<model>_<experiment>.cvdp_data.<start>-<end>.nc` is the main file; the
    auxiliary per-variable outputs (`.siconc.`, `.zos.`, `.monsoon.`, `.tas.indices.`)
    put an extra token before the year range and hold no `tas_spatialmean_ann`.
    """
    out = {}
    pat = os.path.join(CVDP_DIR, experiment, f'*_{experiment}.cvdp_data.*.nc')
    main = re.compile(rf'^(?P<model>.+)_{re.escape(experiment)}\.cvdp_data\.\d+-\d+\.nc$')
    for f in sorted(glob.glob(pat)):
        m = main.match(os.path.basename(f))
        if m:
            out[m.group('model')] = f
    return out

def has_var(path):
    """True if the CVDP file actually carries the tas field (a few don't)."""
    with xr.open_dataset(path, decode_times=False) as ds:
        return VAR in ds.variables

exp_files = model_files(EXPERIMENT)
pi_files  = model_files('piControl')
paired = sorted(set(exp_files) & set(pi_files))
# Some CVDP files lack tas_spatialmean_ann entirely — drop those models.
models = [m for m in paired if has_var(exp_files[m]) and has_var(pi_files[m])]
print(f'{len(models)} models with both {EXPERIMENT} and piControl (and a tas field):')
print(models)
missing_pi = sorted(set(exp_files) - set(pi_files))
if missing_pi:
    print(f'{EXPERIMENT} models with no piControl (skipped):', missing_pi)
no_var = [m for m in paired if m not in models]
if no_var:
    print(f'models dropped — no {VAR} in CVDP file:', no_var)

16 models with both lgm and piControl (and a tas field):
['AWI-ESM-1-1-LR', 'CCSM4', 'CESM2-FV2', 'CESM2-WACCM-FV2', 'CNRM-CM5', 'COSMOS-ASO', 'FGOALS-g2', 'GISS-E2-R', 'INM-CM4-8', 'IPSL-CM5A-LR', 'MIROC-ES2L', 'MIROC-ESM', 'MPI-ESM-P', 'MPI-ESM1-2-LR', 'MRI-CGCM3', 'UofT-CCSM-4']


In [4]:
def load_field(path):
    da = xr.open_dataset(path, decode_times=False)[VAR].sortby('lat').sortby('lon')
    # A few CVDP grids (e.g. LOVECLIM piControl) carry duplicate lon values,
    # which break interpolation; keep the first occurrence of each coordinate.
    for dim in ('lat', 'lon'):
        _, idx = np.unique(da[dim].values, return_index=True)
        if len(idx) != da.sizes[dim]:
            da = da.isel({dim: np.sort(idx)})
    return da

anomalies = {}
for m in models:
    exp_field = load_field(exp_files[m])
    pi  = load_field(pi_files[m])
    if exp_field.shape != pi.shape or not (np.allclose(exp_field.lat, pi.lat) and np.allclose(exp_field.lon, pi.lon)):
        pi = pi.interp(lat=exp_field.lat, lon=exp_field.lon)
    anomalies[m] = (exp_field - pi).rename('tas_anom')
    print(f'{m:18s} grid {exp_field.shape}  mean anom {float(anomalies[m].mean()):+.2f} C')

AWI-ESM-1-1-LR     grid (96, 192)  mean anom -5.12 C


CCSM4              grid (192, 288)  mean anom -6.79 C
CESM2-FV2          grid (96, 144)  mean anom -8.55 C


CESM2-WACCM-FV2    grid (96, 144)  mean anom -9.12 C


CNRM-CM5           grid (128, 256)  mean anom -3.43 C
COSMOS-ASO         grid (48, 96)  mean anom -7.12 C


FGOALS-g2          grid (60, 128)  mean anom -5.93 C


...

IPSL-CM5A-LR       grid (96, 96)  mean anom -5.86 C
MIROC-ES2L         grid (64, 128)  mean anom -5.42 C


MIROC-ESM          grid (64, 128)  mean anom -7.09 C
MPI-ESM-P          grid (96, 192)  mean anom -6.15 C


MPI-ESM1-2-LR      grid (96, 192)  mean anom -5.21 C


MRI-CGCM3          grid (160, 320)  mean anom -6.39 C


UofT-CCSM-4        grid (192, 288)  mean anom -7.19 C


## Step 3 — Sample model anomalies at reconstruction locations

Model longitudes run 0–360°, the proxy longitudes −180–180°, so targets are wrapped to
0–360 and the field is made cyclic in longitude before bilinear interpolation.

Each recon point carries a `weight` used later in the RMSE. Scattered-site compilations
weight every point equally (1.0); gridded near-global reconstructions (Cleator, Osman,
Erb, Tierney) set `weight = cos(latitude)` so the RMSE is area-fair rather than
pole-heavy.

In [5]:
def sample_points(field, lats, lons):
    """Bilinearly sample a (lat, lon) field at scattered points; lon made cyclic."""
    lon_cyc = np.append(field.lon.values, field.lon.values[0] + 360.0)
    fcyc = xr.concat([field, field.isel(lon=0)], dim='lon').assign_coords(lon=lon_cyc)
    ta = xr.DataArray(np.asarray(lats), dims='point')
    to = xr.DataArray(np.asarray(lons) % 360.0, dims='point')
    return fcyc.interp(lat=ta, lon=to).values

sampled = recon[['compilation', 'reference', 'site', 'Proxy',
                 'Latitude', 'Longitude', 'Anom']].copy()
sampled = sampled.rename(columns={'Anom': 'recon_anom'})
# Optional per-point weight (defaults to equal weighting when Step 1 omits it).
sampled['weight'] = recon['weight'].values if 'weight' in recon.columns else 1.0
for m in models:
    sampled[m] = sample_points(anomalies[m], sampled['Latitude'].values, sampled['Longitude'].values)

sampled_out = os.path.join(OUT_DIR, f'model_anom_at_recon_{EXPERIMENT}.csv')
sampled.to_csv(sampled_out, index=False)
print('wrote', sampled_out, '  shape', sampled.shape)
sampled.head()

wrote /home/ucfaccb@ad.ucl.ac.uk/Documents/local_repos/PMIP7-vision/carpet_diagram/output/model_anom_at_recon_lgm.csv   shape (32452, 24)


,compilation,reference,site,Proxy,Latitude,Longitude,recon_anom,weight,AWI-ESM-1-1-LR,CCSM4,...,FGOALS-g2,GISS-E2-R,INM-CM4-8,IPSL-CM5A-LR,MIROC-ES2L,MIROC-ESM,MPI-ESM-P,MPI-ESM1-2-LR,MRI-CGCM3,UofT-CCSM-4
0,Bartlein,Bartlein et al. 2011,NaN,pollen MAT,69.0,-161.0,2.3591,1.0,-2.505068,-5.801219,...,-6.542780,-8.836830,-1.492071,-5.488791,-6.012330,-8.806610,-6.082855,-2.426804,-5.159873,-6.148995
1,Bartlein,Bartlein et al. 2011,NaN,pollen MAT,67.0,-157.0,-3.0103,1.0,-1.179381,-4.217703,...,-4.955228,-7.279610,-0.214015,-4.747612,-4.068210,-7.405939,-2.677890,-1.663227,-3.429791,-3.718091
2,Bartlein,Bartlein et al. 2011,NaN,pollen MAT,67.0,-153.0,-4.4026,1.0,-1.261294,-5.424536,...,-4.251140,-8.293301,-1.332156,-5.804151,-4.004301,-7.545178,-3.727498,-2.500885,-4.116196,-4.696515
3,Bartlein,Bartlein et al. 2011,NaN,pollen MAT,67.0,-149.0,13.3879,1.0,-1.999674,-6.765273,...,-3.901543,-9.720577,-2.049146,-6.066072,-4.089089,-7.874986,-5.373007,-3.138571,-4.640459,-3.686671
4,Bartlein,Bartlein et al. 2011,NaN,pollen MAT,65.0,-147.0,-3.3921,1.0,-0.968812,-5.798272,...,-3.504157,-5.052363,-0.837013,-5.493895,-3.750899,-8.156405,-2.855837,-3.122754,-3.716482,-3.113100


## Step 4 — RMSE of each model against each compilation

For every model × compilation we take the model-minus-proxy difference across all proxy
points in that compilation and report the (weight-weighted) RMSE and bias, plus the number
of points contributing (a point off the model grid edge can return NaN). With unit weights
this is the ordinary RMSE; gridded compilations use `cos(latitude)` weights (Step 3).

In [6]:
records = []
for m in models:
    for comp, grp in sampled.groupby('compilation'):
        diff = grp[m].values - grp['recon_anom'].values
        w = grp['weight'].values
        valid = np.isfinite(diff) & np.isfinite(w)
        n = int(valid.sum())
        if n:
            dv, wv = diff[valid], w[valid]
            rmse = float(np.sqrt(np.sum(wv * dv ** 2) / np.sum(wv)))
            bias = float(np.sum(wv * dv) / np.sum(wv))
        else:
            rmse = bias = np.nan
        records.append({'model': m, 'compilation': comp, 'n_points': n,
                        'rmse': rmse, 'bias': bias})

rmse_long = pd.DataFrame(records)
rmse_wide = rmse_long.pivot(index='model', columns='compilation', values='rmse')
rmse_wide.columns = [f'{c}_RMSE' for c in rmse_wide.columns]

rmse_long.to_csv(os.path.join(OUT_DIR, f'rmse_long_{EXPERIMENT}.csv'), index=False)
rmse_wide.to_csv(os.path.join(OUT_DIR, f'rmse_summary_{EXPERIMENT}.csv'))
print(f'wrote rmse_long_{EXPERIMENT}.csv and rmse_summary_{EXPERIMENT}.csv')
rmse_wide.sort_values(rmse_wide.columns[0])

wrote rmse_long_lgm.csv and rmse_summary_lgm.csv


,Annan_RMSE,Bartlein_RMSE,Cleator_RMSE,Osman_RMSE,P2F_RMSE
model,,,,,
IPSL-CM5A-LR,1.198180,4.481072,2.293258,4.423680,2.206651
MPI-ESM-P,1.365551,5.205005,2.680045,4.104728,2.124899
MIROC-ES2L,1.679015,4.836742,2.691053,4.361049,2.060754
MPI-ESM1-2-LR,1.781511,4.642391,2.970425,4.745336,2.216545
COSMOS-ASO,1.871631,4.870245,2.704134,3.747196,2.788162
AWI-ESM-1-1-LR,1.960376,4.792889,3.537903,4.902935,2.231917
INM-CM4-8,2.109322,4.625708,2.386247,5.005875,2.579646
FGOALS-g2,2.155794,5.064723,2.796614,3.961142,2.379394
CCSM4,2.179536,5.349694,2.780847,3.663350,2.348663


These RMSE values are the building blocks of the carpet diagram: one column per model,
one row per (period, reconstruction compilation), coloured by RMSE. `carpet_figure.py`
reads every `output/rmse_long_<period>.csv`, so re-running this notebook for a new period
makes its rows appear in the portrait plot automatically.